# AQI Prediction and Analysis

This walkthrough uses the project's current date-based evaluation and saved model pipelines.

> **Data limitation:** The bundled dataset is synthetic, and AQI is calculated from the pollutant inputs. Results demonstrate the workflow, not validated performance on real monitors. The model estimates an AQI scenario from supplied measurements; it does not forecast future pollution.

## Walkthrough
1. Inspect the raw records and missingness.
2. Create a chronological training/test split.
3. Train and compare regression and classification models using expanding date folds.
4. Inspect per-category performance and a sample scenario estimate.


In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from preprocess import clean_data, load_and_inspect_raw_data, prepare_and_split_data
from train_models import train_and_evaluate_all
from predict import AQIPredictor

print(f'Project folder: {PROJECT_ROOT}')


## 1. Inspect the raw dataset

The raw CSV includes duplicate records and missing sensor readings. Model preprocessing keeps those readings missing until each training fold fits its own grouped medians.


In [ ]:
raw_df = load_and_inspect_raw_data('data/air_quality_data_raw.csv')
raw_df.head()


## 2. Build a chronological holdout

The latest 20% of dates form the test period. This evaluates whether models trained on earlier dates perform on a later period. All stations recorded on the same date stay together.


In [ ]:
clean_df = clean_data(raw_df, impute=False)
split = prepare_and_split_data(clean_df, test_size=0.2)

train_dates = pd.to_datetime(split['X_train']['Date'])
test_dates = pd.to_datetime(split['X_test']['Date'])
print(f"Training: {len(split['X_train']):,} rows, {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Testing:  {len(split['X_test']):,} rows, {test_dates.min().date()} to {test_dates.max().date()}")


## 3. Train and compare models

Each model is a pipeline that fits grouped median imputation, feature creation, categorical encoding, and scaling on its current training fold. Five expanding date folds are used for validation; then the final model is refit on the full training period and scored on the later test period.

This cell retrains the project models and refreshes the saved model and metric artifacts.


In [ ]:
results = train_and_evaluate_all()


In [ ]:
regression = pd.DataFrame(results['reg_results']).T
regression[['CV_R2_Mean', 'CV_R2_Std', 'Test_R2', 'Test_RMSE', 'Test_MAE']].sort_values('Test_R2', ascending=False)


In [ ]:
classification = pd.DataFrame(results['clf_results']).T
classification[['CV_Accuracy_Mean', 'CV_Accuracy_Std', 'Test_Accuracy', 'Test_F1_Weighted', 'Test_F1_Macro']].sort_values('Test_F1_Macro', ascending=False)


## 4. Inspect category-level results

Macro F1 gives each category equal weight, while weighted F1 gives more weight to common categories. Per-class recall and support help reveal when an overall score hides limited performance on a rare category.


In [ ]:
rf_per_class = results['clf_results']['Random Forest Classifier']['Per_Class']
pd.DataFrame(rf_per_class).T.rename_axis('AQI category')


In [ ]:
display(Image(filename='plots/results/confusion_matrix_classifier.png'))


## 5. Try an AQI scenario estimate

The saved pipelines accept raw measurements and apply the same preprocessing used during training. The displayed class probability is not calibrated, and the advisory text is illustrative rather than an official public-health recommendation.


In [ ]:
predictor = AQIPredictor()
scenario = {
    'Date': '2024-12-10', 'Station': 'Station_North', 'Season': 'Winter',
    'PM2.5': 285.0, 'PM10': 460.0, 'NO2': 110.0, 'SO2': 48.0,
    'CO': 4.2, 'O3': 18.0, 'Temperature': 11.5, 'Humidity': 78.0,
    'Wind_Speed': 3.2
}
predictor.predict(scenario)


## Conclusion and next validation step

This project now uses leakage-safe preprocessing and date-based evaluation. Because the data generator derives AQI from the pollutant inputs, the next step for real-world claims is evaluation on independent, documented monitoring data and comparison with the official AQI calculation. If the goal changes to forecasting, define a future horizon and train against future AQI using only earlier information.
